<div style="background-color: #1A5276; padding: 20px; border-radius: 10px; text-align: center; margin-bottom: 30px;">
    <h1 style="color: white; margin: 0;">Faculty AI — Class Participation Scoring Lab</h1>
    <h2 style="color: white; margin-top: 15px;">Score a day's participation from a Zoom transcript</h2>
    <p style="color: white; margin-top: 10px; font-style: italic;">Grounded in your lesson material · A draft you review</p>
</div>

## What this notebook does

Point it at a **Zoom transcript** of one class meeting and the **lesson material** you covered that day. It will:

1. Figure out **who spoke** and **how much** (turns, words, talk time).
2. Judge **what they talked about** — on-topic vs. off-topic — grounded *only* in your lesson material.
3. Draft a **0–4 participation score** for each student for that day.

The scoring logic, in one line: **talk a lot and on-topic → high; talk a lot but off-topic → low; don't speak → 0.**

## The contract (read this first)

> ⚠️ **This produces a DRAFT score that you review — not a grade the tool assigns on its own.**
>
> - Every score comes with the **evidence quotes** and **reasoning** behind it, plus a **confidence flag**. You check them in Part 5 and adjust before anything reaches a gradebook.
> - Class recordings and transcripts are **student records**. Make sure recording + transcription is covered by your institution's policy and that students were notified (FERPA in the US, or your local equivalent). Keep transcripts on approved storage.
> - Participation is **one signal of engagement, not the only one.** A quiet student or a multilingual student who is processing in a second language is not a disengaged student. Part 7 says more.

It runs in the same SageMaker Studio + Amazon Bedrock environment as the curriculum lab, and reuses the same embedding + retrieval stack.

---
# Part 1 — Setup *(~2 min)*

### 1.1 Install dependencies
Same pinned packages as the curriculum lab — no new dependencies. The transcript parser uses only Python's standard library.

In [ ]:
%%capture
!pip install -q -r requirements.txt

### 1.2 Import tools and connect to Bedrock
One handshake to Bedrock, then we load the participation helpers from `mlu_utils/`.

In [ ]:
import boto3
import csv
import warnings
from datetime import datetime
from IPython.display import Markdown, display

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter

from mlu_utils.embeddings import NovaMultimodalEmbeddings
from mlu_utils.transcript_tools import (
    parse_transcript,
    aggregate_by_speaker,
    load_roster,
    reconcile_roster,
    ParticipationScorer,
    build_report,
)

warnings.filterwarnings("ignore")

bedrock_runtime = boto3.client(service_name="bedrock-runtime", region_name="us-east-1")
embeddings = NovaMultimodalEmbeddings(client=bedrock_runtime)

print("Ready.")

---
# Part 2 — Load the day's lesson material *(~1 min)*

This is the step that makes "on topic" mean something. We embed the material you actually taught that day, so the AI judges each student's talk against **your lesson**, not against the open internet.

### 2.1 🟢 EDIT ME — point to the day's material
Any text-based PDF of what you covered: lecture notes, the reading, slides exported to PDF, the chapter. The sample is Ch. 1 of *Open Data Structures* (the CS Data Structures class in the sample transcript).

In [ ]:
LESSON_PDF = "data/persona2_cs_data_structures.pdf"  # <- change to your day's material

### 2.2 Build the searchable lesson index
Loads, chunks, and embeds the material. Same pattern as the curriculum lab.

In [ ]:
pages = PyPDFLoader(LESSON_PDF).load()
print(f"Loaded {len(pages)} pages from {LESSON_PDF}")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=60,
    separators=["\n\n", "\n", "(?<=\\. )", " ", ""],
    is_separator_regex=True,
)
chunks = splitter.split_documents(pages)
print(f"Split into {len(chunks)} chunks.")

print("Building searchable index... (30-90 seconds)")
vectordb = FAISS.from_documents(chunks, embeddings)
retriever = vectordb.as_retriever(search_kwargs={"k": 6})
print("Ready. This is what \"on topic\" means today.")

---
# Part 3 — Load the transcript and roster *(~1 min)*

Zoom exports a transcript under **Recording → Audio Transcript** as a `.vtt` file (or a `.txt`). Both work. The roster is optional but recommended — without it, a student who never speaks is invisible and can't be scored 0.

### 3.1 🟢 EDIT ME — your transcript and roster
Set `ROSTER_PATH = None` if you don't have a roster CSV. A roster CSV just needs a `student_name` column.

In [ ]:
TRANSCRIPT_PATH = "data/sample_class_transcript.vtt"  # <- your Zoom .vtt or .txt
ROSTER_PATH     = "data/sample_roster.csv"            # <- your roster CSV, or None

### 3.2 Who spoke, and how much
This is **volume only** — not a score yet. Scoring comes in Part 4.

In [ ]:
utterances = parse_transcript(TRANSCRIPT_PATH)
stats = aggregate_by_speaker(utterances)
roster_names = load_roster(ROSTER_PATH) if ROSTER_PATH else None
matched, silent, off_roster = reconcile_roster(stats, roster_names)

rows = ["| Student | Turns | Words | Talk time |", "|---|:---:|:---:|:---:|"]
for name, s in sorted(matched, key=lambda kv: -kv[1].words):
    rows.append(f"| {name} | {s.turns} | {s.words} | {int(s.talk_seconds)}s |")
display(Markdown("### Who spoke today (volume only — not a score)\n" + "\n".join(rows)))

if silent:
    display(Markdown("**Silent** (on the roster, never spoke): " + ", ".join(silent) + " — these will be scored **0**."))
if off_roster:
    display(Markdown("**Not on the roster** (instructor / TA / guest — *not* scored): " + ", ".join(s.name for s in off_roster)))

> 💡 Notice volume alone is misleading: the student with the most words isn't automatically the best participant, and a student with very few words may have said something sharp. Part 4 is where on-topic substance separates them.

---
# Part 4 — Score participation *(~1–2 min)*

For each student who spoke, the AI retrieves the part of your lesson their words relate to, then scores them 0–4 with evidence. Silent students are set to **0** in code — their words never go to the model.

In [ ]:
scorer = ParticipationScorer(retriever, bedrock_runtime)
scored = scorer.score_all(matched, silent)

report_md, csv_rows = build_report(scored)
display(Markdown("## Draft participation scores — " + datetime.now().strftime("%Y-%m-%d") + "\n\n" + report_md))

detail = "\n".join(
    f"- **{s.name}** — tier {s.tier} ({s.confidence}): {s.rationale}"
    for s in scored if s.status == "scored"
)
display(Markdown("### Why each score\n" + detail))

> 💡 **Pause here and read the evidence.** For each student, do the quote and rationale support the tier? The scoring rubric:
>
> | Tier | Means |
> |:---:|---|
> | **4** | Substantive **and** on-topic leadership — probing questions, reasoning, building on peers |
> | **3** | Solid on-topic contribution, but not driving the discussion |
> | **2** | Some on-topic participation, possibly brief or surface-level |
> | **1** | Minimal or off-topic — filler, logistics, or social chatter (even if a lot of it) |
> | **0** | Did not participate |

---
# Part 5 — Review and adjust *(the important part)*

The AI proposes; **you decide.** You were in the room. Override any score you disagree with before you export.

### 5.1 🟢 EDIT ME — override any scores
Add `"Student Name": tier` for anything you want to change, then run the cell.

In [ ]:
OVERRIDES = {
    # "Sofia Reyes": 2,   # example: you recall she made an on-topic point the transcript garbled
}

for s in scored:
    if s.name in OVERRIDES:
        s.tier = OVERRIDES[s.name]
        s.confidence = "instructor-set"
        s.rationale = "Adjusted by instructor after review."

scored.sort(key=lambda s: (-(s.tier if s.tier is not None else -1), -s.words))
report_md, csv_rows = build_report(scored)
display(Markdown("## Final scores (after your review)\n\n" + report_md))

### 5.2 A 30-second equity check
Before you export, look at the bottom of the list:

- Is anyone scored low who you know was **engaged but quiet** — listening, taking notes, processing in a second language? Participation is not the same as talking. Adjust above if your course rewards listening too.
- Did the **transcript garble** anyone (heavy accent, bad mic, cross-talk)? Transcription error should never cost a student points — override it.
- Are you comfortable defending each score to the student it belongs to? If not, change it.

---
# Part 6 — Export your records *(~30 sec)*

In [ ]:
stamp = datetime.now().strftime("%Y%m%d_%H%M")
md_name = f"participation_report_{stamp}.md"
csv_name = f"participation_scores_{stamp}.csv"

with open(md_name, "w") as f:
    f.write(f"# Participation report — {datetime.now().strftime('%Y-%m-%d')}\n\n")
    f.write(f"- Lesson material: `{LESSON_PDF}`\n")
    f.write(f"- Transcript: `{TRANSCRIPT_PATH}`\n")
    f.write(f"- Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}\n\n")
    f.write("_Draft scores reviewed and adjusted by the instructor._\n\n")
    f.write(report_md + "\n")

with open(csv_name, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(csv_rows[0].keys()))
    writer.writeheader()
    writer.writerows(csv_rows)

print(f"Saved {md_name} and {csv_name}")
print("Download them from the file browser on the left.")

---
# Part 7 — Where this does NOT belong

Keep this tool in its lane:

- **Not a sole grade.** Use it as one input you review, alongside written work, projects, office-hours engagement, and the listening that never shows up in a transcript.
- **Not a surveillance tool.** Scoring every word a student says, every class, changes the room. Be transparent with students about what you're doing and why.
- **Not a verdict on a person.** A 1 on one day's transcript is a data point about one class, not a judgment of the student. People have off days, slow days, and quiet-but-thinking days.
- **Not better than the transcript it reads.** Auto-transcription mishears names, drops quiet voices, and scrambles cross-talk. Garbage in, garbage out — which is exactly why Part 5 exists.

**The honest framing for students:** "I use a tool to help me keep track of who contributed to discussion, and I review every entry myself." That sentence should always be true.

---

### Reuse this any day
Change `LESSON_PDF` and `TRANSCRIPT_PATH` and re-run. Each class meeting takes about two minutes end to end.